# 05 · Cuantización de la ISI

**Objetivo.** Elegir los anchos de palabra del FIR de la ISI en punto fijo (paso C3.1 de la guía), con un criterio numérico:

- partir de la decisión 32: entrada `ac_fixed<14, 1>` (ADC), coeficientes `ac_fixed<18, 1>` (puerto de 18 bits del DSP48E2), productos a precisión completa, acumulador con log₂K bits de crecimiento y salida de 14 bits (DAC);
- barrer los bits de los **coeficientes** y del **acumulador**;
- separar las fuentes de error: **referencia** (FIR en flotante con la entrada sin cuantizar) → **flotante con la entrada cuantizada** → **punto fijo**;
- comprobar el criterio de la decisión 27 para la ISI: NRMSE ≤ 1e-3 contra la salida en flotante.

El modelo bit a bit es `emulador/fixed_point/isi_fir.py` (`isi_fir_fixed`), construido sobre `quantization.py` y aritmética entera, con la semántica de `ac_fixed` (tests: `tests/test_fixed_point_isi.py`; el cruce Python ↔ HALCON está en `tests/test_quant_crosscheck.py`, C3.0). La CD (C3.2) se agrega después en este mismo notebook.

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np

from emulador import plots
from emulador.blocks import ISI
from emulador.config import Scenario
from emulador.fixed_point import isi_fir_fixed
from emulador.isi import isi_apply
from emulador.metrics import nrmse
from emulador.quantization import quantize
from emulador.stimuli import comm_signal

plots.use_style()

## 1. Modelo en punto fijo

$$
y[n] = \sum_{k=0}^{K-1} h[k]\,x[n-k]
$$

| Punto | Formato de partida (decisión 32) | Modo (decisión 26) |
|---|---|---|
| Entrada x (ADC) | `<14, 1>`: paso 2⁻¹³, rango [−1, 1) | `AC_RND_CONV`, `AC_SAT` |
| Coeficientes h[k] | `<18, 1>`: paso 2⁻¹⁷ | `AC_RND_CONV`, `AC_SAT` (+1 no es representable: satura a 1 − 2⁻¹⁷) |
| Productos reales | `<32, 2>`, exactos | — |
| Acumulador | `<W_acc, I_acc>`, I_acc = 1 + 1 + 1 + ⌈log₂K⌉ (productos, suma compleja, crecimiento) | precisión completa (30 bits fraccionarios) o recortado a F bits por producto |
| Salida (DAC) | 14 bits, `<14, I_out>` | un solo redondeo `AC_RND_CONV`, `AC_SAT` |

La decisión 32 deja abierto I_out: la sección 2 muestra que con `<14, 1>` la salida satura.

**Fuentes de error** (cada NRMSE contra la referencia, salvo el interno):

- *entrada*: FIR flotante con la entrada cuantizada a 14 bits;
- *coeficientes*: FIR flotante con h cuantizado;
- *salida*: la referencia cuantizada al formato de salida;
- *interno* (lo que agrega el bloque): punto fijo contra el FIR flotante **con la misma entrada cuantizada**;
- *total*: punto fijo con la entrada sin cuantizar contra la referencia.

**Escenario.** 32 GBd, 2 sps (64 GSa/s), 16-QAM con RRC β = 0,1, 20 000 símbolos y 3 semillas (se promedia el NRMSE). Entrada escalada a **pico de componente 0,95**, la convención del PDF de arquitectura de la CD. Perfiles: el principal, Bessel m = 4 con B = 0,6·R_s y K = 16, y el de K = 32 de la regla de la decisión 6, Bessel m = 4 con B = 0,4·R_s.

In [ ]:
esc = Scenario()  # 32 GBd, 2 sps, roll-off 0,1
SEEDS = (1, 2, 3)
N_SYM = 20_000
PEAK = 0.95
X_FMT, H_FMT = (14, 1), (18, 1)
PROFILES = {
    "K = 16 (Bessel, B = 0,6·Rs)": ISI(profile="bessel", B_rel=0.6, order=4, K=16).h,
    "K = 32 (Bessel, B = 0,4·Rs)": ISI(profile="bessel", B_rel=0.4, order=4, K=32).h,
}
CRITERION = 1e-3  # decisión 27, ISI


def signal(seed, peak=PEAK):
    x = comm_signal(esc, "qam16", n_symbols=N_SYM, seed=seed).samples
    return x * peak / np.max(np.abs(np.r_[x.real, x.imag]))


SIGNALS = {s: signal(s) for s in SEEDS}


def mean_over_seeds(f):
    # f(x) -> float o tupla de floats; promedia sobre las semillas
    return np.mean([f(SIGNALS[s]) for s in SEEDS], axis=0)


for name, h in PROFILES.items():
    print(f"{name}: K = {h.size}, Σ|h| = {np.sum(np.abs(h)):.3f}, max|h| = {np.max(np.abs(h)):.3f}")
x = SIGNALS[1]
print(f"señal: {x.size} muestras, rms por componente {np.std(x.real):.3f}, pico/rms {20*np.log10(PEAK/np.std(x.real)):.1f} dB")

## 2. Punto de partida y formato de salida

In [ ]:
def breakdown(x, h, out_fmt, **kw):
    ref = isi_apply(x, h)
    xq = quantize(x, *X_FMT)
    fx_total = isi_fir_fixed(x, h, out_fmt=out_fmt, **kw)
    fx_int = isi_fir_fixed(xq, h, out_fmt=out_fmt, **kw)
    sat = fx_total.saturations
    return (
        nrmse(isi_apply(xq, h), ref),
        nrmse(isi_apply(x, quantize(h, *kw.get("h_fmt", H_FMT))), ref),
        nrmse(quantize(ref, *out_fmt), ref),
        nrmse(fx_int.y, isi_apply(xq, h)),
        nrmse(fx_total.y, ref),
        sat["out"],
        np.max(np.abs(np.r_[ref.real, ref.imag])),
    )


rows = []
print(f"{'perfil':30s} {'salida':8s} {'entrada':>9s} {'coef':>9s} {'salida':>9s} {'interno':>9s} {'total':>9s} {'sat. sal.':>9s} {'pico sal.':>9s}")
for name, h in PROFILES.items():
    for I_out in (1, 2):
        r = mean_over_seeds(lambda x: breakdown(x, h, (14, I_out)))
        rows.append((name, I_out, *r))
        print(f"{name:30s} <14,{I_out}>  " + " ".join(f"{v:9.2e}" for v in r[:5]) + f" {r[5]:9.1f} {r[6]:9.3f}")

**Lectura.**

- Con la entrada a pico 0,95, la salida del Bessel tiene **sobrepicos por encima de 1** (columna "pico sal."). Con salida `<14, 1>` esas muestras saturan (25 a 33 componentes por señal) y el error total queda entre 1,5 y 1,8·10⁻³: **no cumple** el 1e-3 de la decisión 27.
- Con salida `<14, 2>` no satura y el total baja a unos 2·10⁻⁴, dominado por la salida de 14 bits (un bit menos de resolución que `<14, 1>`) y la entrada.
- Los coeficientes de 18 bits y el acumulador de precisión completa aportan un error despreciable: el interno es prácticamente el de la salida.

La alternativa a `<14, 2>` es dejar más margen en la entrada. La sección 3 la mide.

## 3. Escala de pico de la entrada

Error total contra la referencia en función del pico de componente de la entrada, con salida `<14, 1>` y `<14, 2>`. Menos pico deja margen para los sobrepicos de la salida, pero cada componente usa menos del rango del ADC.

In [ ]:
PEAKS = np.array([0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0 - 2**-13])
h16 = PROFILES["K = 16 (Bessel, B = 0,6·Rs)"]
peak_sweep = {}
for I_out in (1, 2):
    vals = []
    for p in PEAKS:
        sig = {s: SIGNALS[s] * p / PEAK for s in SEEDS}
        e = np.mean([nrmse(isi_fir_fixed(sig[s], h16, out_fmt=(14, I_out)).y, isi_apply(sig[s], h16)) for s in SEEDS])
        n_sat = np.mean([isi_fir_fixed(sig[s], h16, out_fmt=(14, I_out)).saturations["out"] for s in SEEDS])
        vals.append((e, n_sat))
    peak_sweep[I_out] = np.array(vals)

fig, ax = plt.subplots(figsize=(6.5, 4))
for I_out, v in peak_sweep.items():
    ax.semilogy(PEAKS, v[:, 0], "o-", label=f"salida <14, {I_out}>")
ax.axhline(CRITERION, color="k", ls="--", lw=1, label="criterio 1e-3 (decisión 27)")
ax.set_xlabel("pico de componente de la entrada")
ax.set_ylabel("NRMSE total")
ax.set_title("ISI K = 16: error total contra el pico de entrada")
ax.legend()
plt.show()

print("pico   " + "  ".join(f"<14,{I}>: NRMSE  sat." for I in peak_sweep))
for i, p in enumerate(PEAKS):
    print(f"{p:.3f}  " + "  ".join(f"{peak_sweep[I][i, 0]:9.2e} {peak_sweep[I][i, 1]:5.1f}" for I in peak_sweep))

**Lectura.**

- Con `<14, 1>` el error es mínimo con pico de entrada 0,85 (1,4·10⁻⁴, mejor que `<14, 2>` con 0,95, porque la salida conserva el bit de resolución). Pero el margen es angosto: con 0,9 ya saturan unas pocas muestras y el error se triplica, y con 0,95 no cumple. El pico de salida depende del perfil (sobrepico del Bessel de 1,086 respecto del pico de entrada); un perfil con más sobrepico, como el Butterworth de estrés, necesitaría todavía menos pico.
- Con `<14, 2>` no satura en todo el barrido y el error baja poco a poco con el pico (de 3,8·10⁻⁴ a 1,9·10⁻⁴). Es la opción robusta: no depende del perfil cargado.
- Las dos cumplen el 1e-3 si la escala está bien elegida. La decisión de formato de salida y de escala de pico es del equipo (sección 6).

## 4. Bits de los coeficientes

Barrido de W_h con I_h = 1 (paso 2^−(W_h−1)), entrada `<14, 1>`, acumulador de precisión completa y salida `<14, 2>`. Se grafica:

- el NRMSE de los propios coeficientes, ‖h_q − h‖ / ‖h‖;
- el error interno con salida de 14 bits (incluye el piso de la salida);
- la contribución de los coeficientes sola: FIR flotante con h cuantizado contra h exacto, con la misma entrada.

In [ ]:
OUT_FMT = (14, 2)
W_HS = np.arange(8, 21)
coef_sweep = {}
for name, h in PROFILES.items():
    vals = []
    for Wh in W_HS:
        hq = quantize(h, Wh, 1)
        e_h = nrmse(hq, h)
        e_int = mean_over_seeds(lambda x: nrmse(isi_fir_fixed(quantize(x, *X_FMT), h, h_fmt=(Wh, 1), out_fmt=OUT_FMT).y, isi_apply(quantize(x, *X_FMT), h)))
        e_coef = mean_over_seeds(lambda x: nrmse(isi_apply(quantize(x, *X_FMT), hq), isi_apply(quantize(x, *X_FMT), h)))
        vals.append((e_h, e_int, e_coef))
    coef_sweep[name] = np.array(vals)

fig, axs = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (name, v) in zip(axs, coef_sweep.items()):
    ax.semilogy(W_HS, v[:, 0], "s-", label="coeficientes ‖h_q − h‖/‖h‖")
    ax.semilogy(W_HS, v[:, 1], "o-", label="interno (salida <14, 2>)")
    ax.semilogy(W_HS, v[:, 2], "^-", label="solo coeficientes")
    ax.axhline(CRITERION, color="k", ls="--", lw=1, label="criterio 1e-3")
    ax.axvline(18, color="0.6", ls=":", lw=1)
    ax.set_xlabel("W_h [bits] (I_h = 1)")
    ax.set_title(name)
axs[0].set_ylabel("NRMSE")
axs[0].legend(fontsize=8)
plt.show()

for name, v in coef_sweep.items():
    floor = v[-1, 1]
    ok = W_HS[v[:, 1] <= CRITERION]
    near = W_HS[v[:, 1] <= 1.1 * floor]
    print(f"{name}: piso interno {floor:.2e}; W_h mínimo con interno ≤ 1e-3: {ok.min()}; con interno ≤ 1,1 × piso: {near.min()}")

## 5. Acumulador

**Crecimiento.** El acumulador reserva I_acc = 3 + ⌈log₂K⌉ bits enteros (incluido el signo). Se compara con lo que la señal usa de verdad: el máximo |parte real o imaginaria| del acumulador.

**Bits fraccionarios.** Con precisión completa el acumulador tiene F = 13 + 17 = 30 bits fraccionarios y es exacto. Si cada producto se recorta a F bits antes de sumar (redondeo a par o truncado), el acumulador es más angosto: W_acc = I_acc + F. El mínimo es F = 12, los bits fraccionarios de la salida `<14, 2>`.

In [ ]:
for name, h in PROFILES.items():
    res = [isi_fir_fixed(SIGNALS[s], h, out_fmt=OUT_FMT) for s in SEEDS]
    W_acc, I_acc = res[0].acc_fmt
    peak = max(r.acc_peak for r in res) * 2.0**-(W_acc - I_acc)
    used = int(np.ceil(np.log2(peak))) + 1 if peak > 0 else 1
    print(f"{name}: acumulador <{W_acc}, {I_acc}> (rango ±{2**(I_acc-1)}); pico usado {peak:.3f} -> {used} bits enteros con signo; margen {I_acc - used} bits")

ACC_FRACS = np.arange(12, 31)
acc_sweep = {}
for name, h in PROFILES.items():
    vals = []
    for F in ACC_FRACS:
        e = [
            mean_over_seeds(lambda x: nrmse(isi_fir_fixed(quantize(x, *X_FMT), h, acc_frac=int(F), acc_rounding=mode, out_fmt=OUT_FMT).y, isi_apply(quantize(x, *X_FMT), h)))
            for mode in ("half_even", "trunc")
        ]
        vals.append(e)
    acc_sweep[name] = np.array(vals)

fig, axs = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (name, v) in zip(axs, acc_sweep.items()):
    I_acc = isi_fir_fixed(SIGNALS[1][:10], PROFILES[name], out_fmt=OUT_FMT).acc_fmt[1]
    ax.semilogy(ACC_FRACS + I_acc, v[:, 0], "o-", label="producto redondeado a par")
    ax.semilogy(ACC_FRACS + I_acc, v[:, 1], "s-", label="producto truncado")
    ax.axhline(CRITERION, color="k", ls="--", lw=1, label="criterio 1e-3")
    ax.set_xlabel(f"W_acc [bits] (I_acc = {I_acc})")
    ax.set_title(name)
axs[0].set_ylabel("NRMSE interno (salida <14, 2>)")
axs[0].legend(fontsize=8)
plt.show()

for name, v in acc_sweep.items():
    I_acc = isi_fir_fixed(SIGNALS[1][:10], PROFILES[name], out_fmt=OUT_FMT).acc_fmt[1]
    floor = v[-1, 0]
    for j, mode in enumerate(("redondeo a par", "truncado")):
        near = ACC_FRACS[v[:, j] <= 1.1 * floor]
        print(f"{name}, {mode}: F mínimo con interno ≤ 1,1 × piso ({floor:.2e}): {near.min()} -> W_acc = {near.min() + I_acc}")

## 6. Resumen y propuesta

Los números de las secciones 2 a 5 (promedio de 3 semillas, 20 000 símbolos, 2 sps):

In [ ]:
for name, h in PROFILES.items():
    r = mean_over_seeds(lambda x: breakdown(x, h, OUT_FMT))
    print(f"{name}, salida <14, 2>, h <18, 1>, acumulador completo:")
    print(f"   entrada 14 bits {r[0]:.2e} | coeficientes {r[1]:.2e} | salida {r[2]:.2e} | interno {r[3]:.2e} | total {r[4]:.2e}")
    print(f"   criterio de la decisión 27 (≤ 1e-3 contra el flotante): interno {'cumple' if r[3] <= CRITERION else 'NO cumple'}, total {'cumple' if r[4] <= CRITERION else 'NO cumple'}")

**Propuesta** (para la fila de formatos de la ISI en §1; la decide Wanda con el equipo):

- **Salida `<14, 2>`** con la entrada a pico 0,95 (como la CD): total 2,0·10⁻⁴ (K = 16) y 1,8·10⁻⁴ (K = 32), sin saturaciones con ningún pico. La alternativa es `<14, 1>` con la entrada a pico 0,85 (1,4·10⁻⁴), con poco margen y dependiente del perfil (sección 3). Con `<14, 1>` y pico 0,95 la salida satura y no cumple el criterio.
- **Coeficientes `<18, 1>`**: entran en el puerto de 18 bits del DSP48E2 y su error (7 a 9·10⁻⁶) queda muy por debajo del piso de la salida (sección 4). El criterio se cumple desde 11 a 12 bits y el piso se alcanza (≤ 1,1 × piso) desde 15 a 16 bits; con menos de 18 se ahorra poco, porque el multiplicador del DSP es el mismo.
- **Acumulador de precisión completa**: `<37, 7>` con K = 16 y `<38, 8>` con K = 32; entra en los 48 bits del acumulador del DSP48E2 sin recortar productos (sección 5). Recortado, alcanza el piso con F = 16 bits fraccionarios si se redondea cada producto (W_acc = 23 y 24) o con 18 a 19 si se trunca. El crecimiento reservado (3 + ⌈log₂K⌉ bits enteros) deja 5 y 6 bits de margen sobre lo que usa la señal: el peor caso Σ|h| (1,5 y 1,7) necesita solo 1 bit más que la entrada, pero achicarlo no ahorra DSP.
- La entrada `<14, 1>` es la del ADC y no se barre.